In [37]:
import numpy as np
from icecream import ic
from dataclasses import dataclass
from tqdm.auto import tqdm
import math


Load of the input and output values from the 'problem_i.npz'

In [38]:
data = np.load('../data/problem_8.npz')
data_X = data['x']
data_Y = data['y']
ic(data_X.shape, data_Y.shape)    

ic| data_X.shape: (6, 50000), data_Y.shape: (50000,)


((6, 50000), (50000,))

I organize the operations by dividing between unitary and binary through dictionaries, so as to call each instruction directly through the dictionary.

In [39]:
BINARY_OPS = {
    'add': lambda a, b: np.add(a, b, where=~np.isnan(a) & ~np.isnan(b)),
    'subtract': lambda a, b: np.subtract(a, b, where=~np.isnan(a) & ~np.isnan(b)),
    'multiply': lambda a, b: np.multiply(a, b, where=~np.isnan(a) & ~np.isnan(b)),
    'divide': lambda a, b: np.divide(a, b, out=np.full_like(a, float('inf')), where=(b != 0) & ~np.isnan(a) & ~np.isnan(b)),
    'power': lambda a, b: np.power(a, b, out=np.full_like(a, float('inf')), where=(a >= 0) & ~np.isnan(a) & ~np.isnan(b))
}

UNARY_OPS = {
    'negate': lambda a: np.negative(a, where=~np.isnan(a)),
    'sin': lambda a: np.sin(np.nan_to_num(a)),
    'cos': lambda a: np.cos(np.nan_to_num(a)),
    'tan': lambda a: np.tan(np.nan_to_num(a)),
    'exp': lambda a: np.exp(np.clip(a, None, 700)),  # Avoid overflow
    'log': lambda a: np.log(np.where(a > 0, a, float('inf'))),
    'sqrt': lambda a: np.sqrt(np.where(a >= 0, a, float('inf'))),
    'abs': lambda a: np.abs(np.nan_to_num(a)),
    'square': lambda a: np.square(np.nan_to_num(a))
}


In [40]:
@dataclass
class TreeNode:
    operator: str | int
    left: object = None
    right: object = None

@dataclass
class Candidate:
    structure: TreeNode
    operation_count: int
    parameters: list
    score: float = None

def clone_tree(node):
    if node is None:
        return None
    return TreeNode(node.operator, clone_tree(node.left), clone_tree(node.right))

def update_tree_value(node, target, replacement):
    if node is None:
        return False
    if isinstance(node.operator, str):
        if node.operator in BINARY_OPS or node.operator in UNARY_OPS:
            return update_tree_value(node.left, target, replacement) or update_tree_value(node.right, target, replacement)
    elif node.operator == target:
        node.operator = replacement
        return True
    return False

def create_random_tree(max_depth, available_values, current_depth=0):
    if current_depth >= max_depth or (current_depth > 0 and np.random.random() < 0.3):
        if available_values:
            value = available_values.pop(np.random.randint(0, len(available_values)))
        else:
            value = -np.random.randint(1, 10)
        return TreeNode(value)
    if np.random.random() > 0.5:
        operation = np.random.choice(list(BINARY_OPS.keys()))
        return TreeNode(operation, 
                        create_random_tree(max_depth, available_values, current_depth + 1),
                        create_random_tree(max_depth, available_values, current_depth + 1))
    else:
        operation = np.random.choice(list(UNARY_OPS.keys()))
        return TreeNode(operation, create_random_tree(max_depth, available_values, current_depth + 1))

def collect_parameters(node, parameters):
    if node is None:
        return 0
    if isinstance(node.operator, str):
        if node.operator in BINARY_OPS or node.operator in UNARY_OPS:
            return collect_parameters(node.left, parameters) + collect_parameters(node.right, parameters)
    elif node.operator < 0:
        parameters.append(node.operator)
        return 1
    return 0

def evaluate_tree(node, input_values, parameters):
    if node is None:
        return 0
    if isinstance(node.operator, str):
        if node.operator in BINARY_OPS:
            left_result = evaluate_tree(node.left, input_values, parameters)
            right_result = evaluate_tree(node.right, input_values, parameters)
            return BINARY_OPS[node.operator](left_result, right_result)
        elif node.operator in UNARY_OPS:
            operand_result = evaluate_tree(node.left, input_values, parameters)
            return UNARY_OPS[node.operator](operand_result)
    elif node.operator < 0:
        return parameters[-node.operator - 1]
    return input_values[node.operator]

def simplify_tree(candidate):
    def simplify_node(node, parameters):
        if node is None:
            return 0
        if isinstance(node.operator, str):
            if node.operator in BINARY_OPS or node.operator in UNARY_OPS:
                return simplify_node(node.left, parameters) + simplify_node(node.right, parameters)
        elif node.operator < 0:
            parameters.append(node.operator)
        return 0

    params = []
    simplify_node(candidate.structure, params)
    candidate.operation_count = count_operations(candidate.structure)
    return candidate

def count_operations(node):
    if node is None:
        return 0
    if isinstance(node.operator, str):
        if node.operator in BINARY_OPS or node.operator in UNARY_OPS:
            return 1 + count_operations(node.left) + count_operations(node.right)
    return 0

def generate_expression_string(node, parameters):
    if node is None:
        return ""
    if isinstance(node.operator, str):
        if node.operator in BINARY_OPS:
            return f"({generate_expression_string(node.left, parameters)} {node.operator} {generate_expression_string(node.right, parameters)})"
        elif node.operator in UNARY_OPS:
            return f"{node.operator}({generate_expression_string(node.left, parameters)})"
    elif node.operator < 0:
        return f"p{(-node.operator) - 1}"
    return f"x{node.operator}"

def compute_fitness(candidate, data_X, data_Y):
    total_error = 0
    for index, row in enumerate(data_X):
        prediction = evaluate_tree(candidate.structure, row, candidate.parameters)
        if not np.isnan(prediction):
            total_error += np.square(data_Y[index] - prediction)
    candidate.score = -total_error / data_X.shape[0]
    return candidate.score

def mutate_candidate(candidate, mutation_rate):
    new_structure = clone_tree(candidate.structure)
    new_parameters = candidate.parameters.copy()

    def mutate_node(node):
        if node is None:
            return
        if isinstance(node.operator, str):
            if node.operator in BINARY_OPS or node.operator in UNARY_OPS:
                if np.random.random() < mutation_rate:
                    node.operator = np.random.choice(list(BINARY_OPS.keys() if node.operator in BINARY_OPS else UNARY_OPS.keys()))
            mutate_node(node.left)
            if node.operator in BINARY_OPS:
                mutate_node(node.right)

    mutate_node(new_structure)
    return Candidate(new_structure, count_operations(new_structure), new_parameters)

def crossover_candidates(parent1, parent2):
    new_structure = clone_tree(parent1.structure)
    crossover_point = new_structure

    while crossover_point is not None and isinstance(crossover_point.operator, str) and crossover_point.operator in UNARY_OPS:
        crossover_point = crossover_point.left

    if crossover_point is not None and isinstance(crossover_point.operator, str) and crossover_point.operator in BINARY_OPS:
        if np.random.random() > 0.5:
            crossover_point.right = clone_tree(parent2.structure.left) if parent2.structure.left else None
        else:
            crossover_point.right = clone_tree(parent2.structure.right) if parent2.structure.right else None

    return Candidate(new_structure, count_operations(new_structure), parent1.parameters + parent2.parameters)

# Ciclo principale per evoluzione
POPULATION_SIZE = 50
GENERATIONS = 100
population = []

while len(population) < POPULATION_SIZE:
    random_tree = create_random_tree(5, [i for i in range(data_X.shape[1])])
    parameters = []
    collect_parameters(random_tree, parameters)
    candidate = Candidate(random_tree, count_operations(random_tree), parameters)
    compute_fitness(candidate, data_X, data_Y)
    population.append(candidate)

for generation in tqdm(range(GENERATIONS)):
    offspring = []
    for _ in range(POPULATION_SIZE // 2):
        parent1 = np.random.choice(population)
        parent2 = np.random.choice(population)

        child1 = crossover_candidates(parent1, parent2)
        child2 = crossover_candidates(parent2, parent1)

        child1 = mutate_candidate(child1, 0.1)
        child2 = mutate_candidate(child2, 0.1)

        compute_fitness(child1, data_X, data_Y)
        compute_fitness(child2, data_X, data_Y)

        offspring.extend([child1, child2])

    population.extend(offspring)
    population = sorted(population, key=lambda c: c.score, reverse=True)[:POPULATION_SIZE]

best_candidate = population[0]
ic(best_candidate.score)
print(generate_expression_string(best_candidate.structure, best_candidate.parameters))


C:\Users\chiar\AppData\Local\Temp\ipykernel_8240\859566087.py:18: RuntimeWarning: overflow encountered in square
  'square': lambda a: np.square(np.nan_to_num(a))
C:\Users\chiar\AppData\Local\Temp\ipykernel_8240\103104188.py:113: RuntimeWarning: overflow encountered in scalar add
  total_error += np.square(data_Y[index] - prediction)
C:\Users\chiar\AppData\Local\Temp\ipykernel_8240\103104188.py:113: RuntimeWarning: overflow encountered in square
  total_error += np.square(data_Y[index] - prediction)
C:\Users\chiar\AppData\Local\Temp\ipykernel_8240\859566087.py:4: RuntimeWarning: invalid value encountered in multiply
  'multiply': lambda a, b: np.multiply(a, b, where=~np.isnan(a) & ~np.isnan(b)),


  0%|          | 0/100 [00:00<?, ?it/s]

C:\Users\chiar\AppData\Local\Temp\ipykernel_8240\859566087.py:3: RuntimeWarning: invalid value encountered in subtract
  'subtract': lambda a, b: np.subtract(a, b, where=~np.isnan(a) & ~np.isnan(b)),
C:\Users\chiar\AppData\Local\Temp\ipykernel_8240\859566087.py:5: RuntimeWarning: overflow encountered in divide
  'divide': lambda a, b: np.divide(a, b, out=np.full_like(a, float('inf')), where=(b != 0) & ~np.isnan(a) & ~np.isnan(b)),
C:\Users\chiar\AppData\Local\Temp\ipykernel_8240\859566087.py:4: RuntimeWarning: overflow encountered in multiply
  'multiply': lambda a, b: np.multiply(a, b, where=~np.isnan(a) & ~np.isnan(b)),
C:\Users\chiar\AppData\Local\Temp\ipykernel_8240\859566087.py:5: RuntimeWarning: invalid value encountered in divide
  'divide': lambda a, b: np.divide(a, b, out=np.full_like(a, float('inf')), where=(b != 0) & ~np.isnan(a) & ~np.isnan(b)),
C:\Users\chiar\AppData\Local\Temp\ipykernel_8240\859566087.py:6: RuntimeWarning: overflow encountered in power
  'power': lambda a

(log((x49945 power log(x20798))) multiply )
